### Load Dependencies and Data

In [ ]:
# Set dependencies
import pandas as pd
import numpy as np
import torch
from pyprojroot import here
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"



In [ ]:
# Load Data

def load_split(dataset: str, split: str) -> pd.DataFrame:
    X = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_X_{split}.csv")
    y = pd.read_csv(PROCESSED_DATA_DIR / f"{dataset}_Y_{split}.csv")
    df = pd.concat([X, y], axis=1)
    df["dataset"] = dataset
    df["split"] = split
    return df

all_data = pd.concat(
    [load_split(ds, sp) for ds in ("multiturn", "singleturn") for sp in ("train", "val", "test")],
    ignore_index=True,
)

all_data["conversation_id"] = all_data["conversation_id"].astype(str)


### Final Check for Balance, Length, and Quality

In [ ]:
# Check overall structure
all_data.info()

In [ ]:
# check for leakage across train, val, test
all_data["conversation_id"].duplicated().sum()
all_data.duplicated(subset="conversation").sum()
all_data.groupby("conversation_id")["split"].nunique().gt(1).sum()


In [ ]:
# nulls or empty last check
all_data.isna().sum()
(all_data["conversation"].str.strip() == "").sum()

In [ ]:
# Label balance look
all_data.groupby(["dataset", "split"])["harm"].mean()
all_data.groupby(["dataset", "split"])["harm"].value_counts(normalize=True).unstack()


In [ ]:
# length and structure by source

all_data["n_chars"] = all_data["conversation"].str.len()
all_data["n_turns"] = all_data["conversation"].str.count("USER:|ASSISTANT:")
all_data.groupby("dataset")[["n_chars", "n_turns"]].describe()



In [ ]:
# save the all_data file
all_data.to_parquet(PROCESSED_DATA_DIR / "all_data.parquet", index=False)


### Generate 

### Generate Embeddings

Multiturn conversations length exceeds the GPT-2 1024-token window so selected a longer context model instead.  Word2Vec doesn't capture positional information which is important in the conversations.  We selected the nomic-ai embedding model because of the length of the data and to ensure we maintain positional context.



#### GPT-2 Embeddings


In [ ]:
# GPT -2 embeddings as baseline

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

out_path_gpt_2 = PROCESSED_DATA_DIR / "all_data_emb_gpt2.npy"

if not out_path_gpt_2.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModel.from_pretrained("gpt2").to(device)
    model.eval()

    def embed_gpt2(texts, batch_size=8):
        all_embeddings = []
        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size)):
                batch = texts[i:i + batch_size]
                inputs = tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=1024,
                ).to(device)

                outputs = model(**inputs)
                hidden_states = outputs.last_hidden_state
                mask = inputs["attention_mask"].unsqueeze(-1)

                summed = (hidden_states * mask).sum(dim=1)
                counts = mask.sum(dim=1).clamp(min=1)
                pooled = summed / counts

                all_embeddings.append(pooled.cpu().numpy())

        return np.concatenate(all_embeddings, axis=0)
    
    embeddings = embed_gpt2(all_data["conversation"].tolist())

    np.save(out_path_gpt_2, embeddings)

#### Qwen 3 Embeddings

In [ ]:
# Qwen 3

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

out_path_qwen_3 = PROCESSED_DATA_DIR / "all_data_emb_qwen3.npy"

if not out_path_qwen_3.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

    mask = (all_data["dataset"] == "singleturn").values

    # encode single turn first
    
    emb_single = model.encode(
        all_data.loc[mask, "conversation"].tolist(),
        batch_size=4,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # embed multi-turn with smaller batch to avoid out of memory errors
    emb_multi = model.encode(
        all_data.loc[~mask, "conversation"].tolist(),
        batch_size=1,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.empty((len(all_data), emb_single.shape[1]), dtype=emb_single.dtype)
    embeddings[mask] = emb_single
    embeddings[~mask] = emb_multi

    np.save(out_path_qwen_3, embeddings)

#### Nomic-AI Embeddings

In [ ]:
out_path = PROCESSED_DATA_DIR / "all_data_emb_nomic.npy"

# check to make sure the file doesn't exist
# warning that this will take a few hours to 
# create depending on processor available

if not out_path.exists():
    # check type of processor available
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"

    # create the model and fix the max input length to the models max
    model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True, device=device)
    model.max_seq_length = 8192

    mask = (all_data["dataset"] == "singleturn").values

    # encode the single turn data with a larger batch size for speed 
    emb_single = model.encode(
        ("classification: " + all_data.loc[mask, "conversation"]).tolist(),
        batch_size=8,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    # encode the multiturn with smaller batch size
    emb_multi = model.encode(
        ("classification: " + all_data.loc[~mask, "conversation"]).tolist(),
        batch_size=1,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.empty((len(all_data), emb_single.shape[1]), dtype=emb_single.dtype)
    embeddings[mask] = emb_single
    embeddings[~mask] = emb_multi

    # save embeddings as np.array file
    np.save(out_path, embeddings)



### Review and check the embeddings

In [ ]:
# load embeddings
emb_gpt2 = np.load(PROCESSED_DATA_DIR / "all_data_emb_gpt2.npy")
emb_qwen3 = np.load(PROCESSED_DATA_DIR / "all_data_emb_qwen3.npy")
emb_nomic = np.load(PROCESSED_DATA_DIR / "all_data_emb_nomic.npy")

In [ ]:
# Shape of the embeddings
print(f"Size of the original data: {len(all_data)}")
print(f"Shape of the GPT2 embeddings: {emb_gpt2.shape}")
print(f"Shape of the QWEN3 embeddings: {emb_qwen3.shape}")
print(f"Shape of the Nomic embeddings: {emb_nomic.shape}")